In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/ICT AI ML/dataset/adult_dataset.csv")

print("Dataset shape:", df.shape)
display(df.head())
print(df.columns.tolist())
print(df.info())

Dataset shape: (48842, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39.0,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40.0,NaN,<=50K
1,50.0,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13.0,United-States,<=50K
2,38.0,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40.0,United-States,<=50K
3,53.0,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40.0,United-States,<=50K
4,28.0,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40.0,Cuba,<=50K


['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             46400 non-null  float64
 1   workclass       45480 non-null  object 
 2   fnlwgt          48842 non-null  int64  
 3   education       48842 non-null  object 
 4   education-num   48842 non-null  int64  
 5   marital-status  48842 non-null  object 
 6   occupation      45485 non-null  object 
 7   relationship    48842 non-null  object 
 8   race            48842 non-null  object 
 9   sex             48842 non-null  object 
 10  capital-gain    48842 non-null  int64  
 11  capital-loss    48842 non-null  int64  
 12  hours-per-week  46400 non-null  float64
 13  native-co

In [ ]:
print(df.columns.tolist())
print(df.tail())

['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
        age     workclass  fnlwgt  education  education-num  \
48837  39.0       Private  215419  Bachelors             13   
48838  64.0           NaN  321403    HS-grad              9   
48839  38.0       Private  374983  Bachelors             13   
48840  44.0       Private   83891  Bachelors             13   
48841  35.0  Self-emp-inc  182148  Bachelors             13   

           marital-status       occupation    relationship  \
48837            Divorced   Prof-specialty   Not-in-family   
48838             Widowed              NaN  Other-relative   
48839  Married-civ-spouse   Prof-specialty         Husband   
48840            Divorced     Adm-clerical       Own-child   
48841  Married-civ-spouse  Exec-managerial         Husband   

                     race     sex  capital-gai

In [ ]:
target_col = "income"

print(df[target_col].value_counts(dropna=False))

income
<=50K     24720
<=50K.    12435
>50K       7841
>50K.      3846
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Clean column names
df.columns = df.columns.str.strip()

# Remove rows with missing target
df = df.dropna(subset=[target_col]).copy()

# Separate features and target
X = df.drop(columns=[target_col])
y = df[target_col].astype(str).str.strip()

# Remove identifier columns if present
id_cols = [
    col for col in X.columns
    if col.lower() in ["id", "person_id"]
]
X = X.drop(columns=id_cols)

# Replace common missing-value markers
X = X.replace("?", np.nan)

# Identify numeric and categorical columns
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(exclude=np.number).columns.tolist()

# Preprocess numeric features
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocess categorical features
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

# Encode target labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Target classes:", label_encoder.classes_)
print("Number of classes:", len(label_encoder.classes_))

Target classes: ['<=50K' '<=50K.' '>50K' '>50K.']
Number of classes: 4


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Convert sparse matrices to dense arrays if needed
if hasattr(X_train_processed, "toarray"):
    X_train_processed = X_train_processed.toarray()
    X_test_processed = X_test_processed.toarray()

# Deep learning frameworks use floating-point input arrays
X_train_processed = X_train_processed.astype("float32")
X_test_processed = X_test_processed.astype("float32")

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

Training shape: (39073, 105)
Testing shape: (9769, 105)


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Clean target column by removing trailing periods and spaces
y_clean = df[target_col].astype(str).str.strip().str.rstrip('.')

# Re-encode clean target labels
label_encoder = LabelEncoder()
y_encoded_clean = label_encoder.fit_transform(y_clean)

if len(label_encoder.classes_) != 2:
    raise ValueError(f"Expected a binary target, but got classes: {label_encoder.classes_}")

# Re-split with the clean target labels
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded_clean,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded_clean
)

# Re-apply preprocessor
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
if hasattr(X_train_processed, "toarray"):
    X_train_processed = X_train_processed.toarray()
    X_test_processed = X_test_processed.toarray()

X_train_processed = X_train_processed.astype("float32")
X_test_processed = X_test_processed.astype("float32")

torch.manual_seed(42)

X_train_t = torch.tensor(X_train_processed, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)

X_test_t = torch.tensor(X_test_processed, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

class IncomeNet(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

torch_model = IncomeNet(X_train_processed.shape[1])

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(torch_model.parameters(), lr=0.001)

for epoch in range(10):
    torch_model.train()
    total_loss = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = torch_model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/10, "
        f"Loss: {total_loss / len(train_loader):.4f}"
    )

Epoch 1/10, Loss: 0.3480
Epoch 2/10, Loss: 0.3134
Epoch 3/10, Loss: 0.3113
Epoch 4/10, Loss: 0.3086
Epoch 5/10, Loss: 0.3069
Epoch 6/10, Loss: 0.3056
Epoch 7/10, Loss: 0.3034
Epoch 8/10, Loss: 0.3026
Epoch 9/10, Loss: 0.3021
Epoch 10/10, Loss: 0.3019


In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout

tf.keras.utils.set_random_seed(42)

tf_model = Sequential([
    Input(shape=(X_train_processed.shape[1],)),
    Dense(64, activation="relu"),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])

tf_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = tf_model.fit(
    X_train_processed,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
489/489 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8332 - loss: 0.3538 - val_accuracy: 0.8522 - val_loss: 0.3197
Epoch 2/10
489/489 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8538 - loss: 0.3173 - val_accuracy: 0.8536 - val_loss: 0.3180
Epoch 3/10
489/489 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8564 - loss: 0.3113 - val_accuracy: 0.8550 - val_loss: 0.3164
Epoch 4/10
489/489 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8587 - loss: 0.3080 - val_accuracy: 0.8545 - val_loss: 0.3162
Epoch 5/10
489/489 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8585 - loss: 0.3063 - val_accuracy: 0.8557 - val_loss: 0.3161
Epoch 6/10
489/489 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8595 - loss: 0.3051 - val_accuracy: 0.8559 - val_loss: 0.3155
Epoch 7/10
489/489 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8596 - loss: 0.3037 - val_accuracy: 0.8540 - val_loss: 0.3165
Epoch 8/10
489/489 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8611 - loss: 0.3022 - val_accuracy: 0.

In [ ]:

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time

# PyTorch predictions
start = time.perf_counter()

torch_model.eval()
with torch.no_grad():
    torch_probs = torch.sigmoid(torch_model(X_test_t)).numpy().ravel()

torch_time = time.perf_counter() - start
torch_preds = (torch_probs >= 0.5).astype(int)

# TensorFlow predictions
start = time.perf_counter()

tf_probs = tf_model.predict(X_test_processed, verbose=0).ravel()

tf_time = time.perf_counter() - start
tf_preds = (tf_probs >= 0.5).astype(int)

def get_metrics(name, actual, predicted, elapsed):
    return {
        "Framework": name,
        "Accuracy": accuracy_score(actual, predicted),
        "Precision": precision_score(actual, predicted, zero_division=0),
        "Recall": recall_score(actual, predicted, zero_division=0),
        "F1-score": f1_score(actual, predicted, zero_division=0),
        "Prediction time (seconds)": elapsed
    }

comparison = pd.DataFrame([
    get_metrics("PyTorch", y_test, torch_preds, torch_time),
    get_metrics("TensorFlow", y_test, tf_preds, tf_time)
])

display(comparison.round(4))

,Framework,Accuracy,Precision,Recall,F1-score,Prediction time (seconds)
0,PyTorch,0.8601,0.7338,0.6518,0.6904,0.0067
1,TensorFlow,0.8590,0.7416,0.6309,0.6818,1.0030


## Conclusion: PyTorch vs TensorFlow

Based on the experimental results, PyTorch performed slightly better than TensorFlow in overall accuracy and F1-score for the income classification task.

PyTorch achieved an accuracy of 86.01% and an F1-score of 0.6904, while TensorFlow achieved an accuracy of 85.90% and an F1-score of 0.6818. TensorFlow had slightly higher precision (0.7416) than PyTorch (0.7338), whereas PyTorch achieved higher recall (0.6518) than TensorFlow (0.6309).

PyTorch also showed a lower measured prediction time of 0.0067 seconds, compared with 1.0030 seconds for TensorFlow. However, prediction time can vary depending on initialization, hardware, batch size, and framework overhead, so repeated measurements are needed for a fair performance comparison.

Overall, PyTorch performed marginally better in this experiment, while TensorFlow achieved slightly higher precision. The results suggest that both frameworks can effectively build deep learning models for binary classification. The better choice depends on the application's requirements, development workflow, and performance needs.